In [2]:
# 필요한 라이브러리 import 및 csv 불러오기
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/extracted_data2.csv',
                 names= ['r.game_id', 'period_id', 'time_seconds', 'team_id','player_id', 'result_name', 'start_x','start_y', 'end_x','end_y', 'dx', 'dy', 'type_name', 'player_name_ko','m.game_id', 'home_team_id', 'away_team_id', 'home_score', 'away_score'])

# NaN행 제거
df = df.dropna(subset=['player_name_ko'])
df = df.dropna(subset=['player_name_ko'])

In [3]:
# 볼 점유율 변수 생성
# df['ball_occupancy'].value_counts()  # 0.000: 130302 -> 보통 한 선수가 계속 볼을 가지고 있을 때의 값
# df[df['ball_occupancy'] != 0]['ball_occupancy'].mean()
df['time_seconds_next'] = df['time_seconds'].shift(periods=-1)
df['ball_occupancy'] = df['time_seconds_next'] - df['time_seconds']
df['ball_occupancy'] = df['ball_occupancy'].fillna(0.007)

In [4]:
# 홈/어웨이 팀 득졈 여부 (일반 골 + 자책골)
df['is_home_goal'] = ((df['team_id'] == df['home_team_id']) & (df['result_name'] == 'Goal')).astype(int)
df['is_away_goal'] = ((df['team_id'] == df['away_team_id']) & (df['result_name'] == 'Goal')).astype(int)

df.loc[(df['team_id'] == df['away_team_id']) & (df['type_name'] == 'Own Goal'), 'is_home_goal'] = 1
df.loc[(df['team_id'] == df['home_team_id']) & (df['type_name'] == 'Own Goal'), 'is_away_goal'] = 1

# 실시간 누적 스코어
df['curr_home_score'] = df.groupby('r.game_id')['is_home_goal'].cumsum()
df['curr_away_score'] = df.groupby('r.game_id')['is_away_goal'].cumsum()

# 이벤트 발생 직전의 점수
df['prev_home_score'] = df.groupby('r.game_id')['curr_home_score'].shift(1, fill_value=0)
df['prev_away_score'] = df.groupby('r.game_id')['curr_away_score'].shift(1, fill_value=0)

# 현재 점수차 계산
def calculate_diff(row):
  if row['team_id'] == row['home_team_id']:
    return row['prev_home_score'] - row['prev_away_score']
  else:
    return row['prev_away_score'] - row['prev_home_score']

df['current_score_diff'] = df.apply(calculate_diff, axis = 1)

In [5]:
# 경기 결과값(target), 승리: 1, 무승부: 0.5, 패배: 0
def label_result(row):
    if row['team_id'] == row['home_team_id']:
        if row['home_score'] > row['away_score']: return 1
        elif row['home_score'] == row['away_score']: return 0.5
        else: return 0
    else:
        if row['away_score'] > row['home_score']: return 1
        elif row['away_score'] == row['home_score']: return 0.5
        else: return 0


df['target'] = df.apply(label_result, axis=1)

In [6]:
# 필요한 변수만 출력
x = ['period_id', 'time_seconds', 'player_id',
       'start_x', 'start_y', 'end_x', 'end_y', 'dx', 'dy',
       'type_name','ball_occupancy',
       'current_score_diff', 'target']
df = df[x]

In [8]:
# 데이터 내보내기
df.to_csv('../data/processed/processed_data.csv')